# Lesson 3: Agentic Search

In [1]:
# libraries
from dotenv import load_dotenv
import os
from tavily import TavilyClient

# load environment variables from .env file
_ = load_dotenv()

# connect
client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

In [2]:
# run search
result = client.search("What is in Nvidia's new Blackwell GPU?",
                       include_answer=True)

# print the answer
result["answer"]


"Nvidia's new Blackwell GPU features 5th Generation Tensor Cores with FP4 precision, offering up to 3x higher AI performance. It includes a 10 TB/s chip-to-chip interconnect and 96GB of GDDR7 memory. The architecture delivers unparalleled parallel computing capabilities."

## Regular search

In [3]:
# choose location (try to change to your own city!)

city = "San Francisco"

query = f"""
    what is the current weather in {city}?
    Should I travel there today?
    "weather.com"
"""

> Note: search was modified to return expected results in the event of an exception. High volumes of student traffic sometimes cause rate limit exceptions.

In [4]:
import requests
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS
import re

ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        print(f"returning previous results due to exception reaching ddg.")
        results = [ # cover case where DDG rate limits due to high deeplearning.ai volume
            "https://weather.com/weather/today/l/USCA0987:1:US",
            "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
        ]
        return results  


for i in search(query):
    print(i)

returning previous results due to exception reaching ddg.
https://weather.com/weather/today/l/USCA0987:1:US
https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8


In [5]:
def scrape_weather_info(url):
    """Scrape content from the given URL"""
    if not url:
        return "Weather information could not be found."
    
    # fetch data
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return "Failed to retrieve the webpage."

    # parse result
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup


> Note: This produces a long output, you may want to right click and clear the cell output after you look at it briefly to avoid scrolling past it.

In [6]:
# use DuckDuckGo to find websites and take the first result
url = search(query)[0]

# scrape first wesbsite
soup = scrape_weather_info(url)

print(f"Website: {url}\n\n")
print(str(soup.body)[:50000]) # limit long outputs

returning previous results due to exception reaching ddg.
Website: https://weather.com/weather/today/l/USCA0987:1:US


<body class="inter_3e69cff8-module__PPczxG__className font-sans"><div hidden=""><!--$--><!--/$--></div><script>(self.__next_s=self.__next_s||[]).push(["/wx-next-web/static/vendor/@mparticle/web-sdk@2.47.1/dist/mparticle.min.js",{}])</script><script>(self.__next_s=self.__next_s||[]).push(["https://weather.com/api/v1/script/dprSdkScript.js",{"id":"dprsdk-script"}])</script><script>(self.__next_s=self.__next_s||[]).push([0,{"children":"\n\t\t\t\t\tif (window.top?.DprSdk) {\n\t\t\t\t\t\tasync function initDprSdk() {\n\t\t\t\t\t\t\ttry {\n\t\t\t\t\t\t\t\tawait window.top?.DprSdk.init({\n\t\t\t\t\t\t\t\t\tgetApplicationInfo: () => ({ id: 'weather.com', version: '2.0.0' }),\n\t\t\t\t\t\t\t\t\tgetUserRegime: () => 'usa-ccpa',\n\t\t\t\t\t\t\t\t});\n\t\t\t\t\t\t\t} catch (_error) {\n\t\t\t\t\t\t\t\t// do nothing.\n\t\t\t\t\t\t\t}\n\t\t\t\t\t\t}\n\t\t\t\t\t\tinitDprSdk();\n\t\t\t

In [7]:
# extract text
weather_data = []
for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
    text = tag.get_text(" ", strip=True)
    weather_data.append(text)

# combine all elements into a single string
weather_data = "\n".join(weather_data)

# remove all spaces from the combined text
weather_data = re.sub(r'\s+', ' ', weather_data)
    
print(f"Website: {url}\n\n")
print(weather_data)

Website: https://weather.com/weather/today/l/USCA0987:1:US


Home Home Forecast Forecast Radar Radar Video Video Explore Explore Home Home Forecast Forecast Radar Radar Video Video Explore Explore Downtown, San Francisco, California Weather San Francisco Today's Outlook Wind Humidity Air Quality Dew Point Pressure UV Index Visibility Moon Phase Sunrise Sunset Things to do around San Francisco Trending Now Two Buttes Reservoir: A Cycle Of Mother Nature 0:46 Summer Outlook: Developing El Niño Could Keep Northeast Cooler As West Sweats Overnight Tornado Rips Through Michigan Ice Arenas, Homes 0:33 Severe Weather, Including Threat Of Strong Tornadoes And Flooding Rain, Forecast Friday In Plains, Midwest Two Buttes Reservoir: A Cycle Of Mother Nature 0:46 Summer Outlook: Developing El Niño Could Keep Northeast Cooler As West Sweats Overnight Tornado Rips Through Michigan Ice Arenas, Homes 0:33 Severe Weather, Including Threat Of Strong Tornadoes And Flooding Rain, Forecast Friday In Plains,

## Agentic Search

In [9]:
# run search
result = client.search(query, max_results=1)

# print first result
data = result["results"][0]["content"]

print(data)

{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1776417889, 'localtime': '2026-04-17 02:24'}, 'current': {'last_updated_epoch': 1776417300, 'last_updated': '2026-04-17 02:15', 'temp_c': 14.4, 'temp_f': 57.9, 'is_day': 0, 'condition': {'text': 'Clear', 'icon': '//cdn.weatherapi.com/weather/64x64/night/113.png', 'code': 1000}, 'wind_mph': 9.6, 'wind_kph': 15.5, 'wind_degree': 10, 'wind_dir': 'N', 'pressure_mb': 1017.0, 'pressure_in': 30.04, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 27, 'cloud': 0, 'feelslike_c': 13.3, 'feelslike_f': 55.9, 'windchill_c': 9.7, 'windchill_f': 49.4, 'heatindex_c': 11.7, 'heatindex_f': 53.1, 'dewpoint_c': 1.5, 'dewpoint_f': 34.8, 'vis_km': 16.0, 'vis_miles': 9.0, 'uv': 0.0, 'gust_mph': 18.2, 'gust_kph': 29.3}}


In [10]:
import json
from pygments import highlight, lexers, formatters

# parse JSON
parsed_json = json.loads(data.replace("'", '"'))

# pretty print JSON with syntax highlighting
formatted_json = json.dumps(parsed_json, indent=4)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())

print(colorful_json)


{
    "location": {
        "name": "San Francisco",
        "region": "California",
        "country": "United States of America",
        "lat": 37.775,
        "lon": -122.4183,
        "tz_id": "America/Los_Angeles",
        "localtime_epoch": 1776417889,
        "localtime": "2026-04-17 02:24"
    },
    "current": {
        "last_updated_epoch": 1776417300,
        "last_updated": "2026-04-17 02:15",
        "temp_c": 14.4,
        "temp_f": 57.9,
        "is_day": 0,
        "condition": {
            "text": "Clear",
            "icon": "//cdn.weatherapi.com/weather/64x64/night/113.png",
            "code": 1000
        },
        "wind_mph": 9.6,
        "wind_kph": 15.5,
        "wind_degree": 10,
        "wind_dir": "N",
        "pressure_mb": 1017.0,
        "pressure_in": 30.04,
        "precip_mm": 0.0,
        "precip_in": 0.0,
        "humidity": 27,
        "cloud": 0,
        "feelslike_c": 13.3,
        "feelslike_f": 55.9,
        "windchill_c": 9.7,
        "windch

<img src="./google_sample.png" width="800" height="600">